# Nepal ANPR — Simple Pipeline (v0)

**Flow:** video file → YOLO plate detection (pretrained) → crop plate → EasyOCR read → log to `plate_log.txt` with timestamp.

This is a starting skeleton, not the final system:
- Uses a generic pretrained license-plate YOLO model (not fine-tuned on Nepali plates yet).
- Uses EasyOCR out of the box (no Devanagari handling, no format validation yet).
- Processes a video file, not a live RTSP stream yet.

Each stage below is its own cell so you can swap detector/OCR later without rewriting everything.

## 1. Install dependencies
Run once per Colab session (fresh VM each time).

In [ ]:
!pip install -q ultralyticsplus==0.0.28 ultralytics==8.0.43
!pip install -q easyocr opencv-python-headless

## 2. Imports

In [ ]:
import cv2
import easyocr
import re
import os
from datetime import datetime, timedelta
from ultralyticsplus import YOLO

## 3. Config
Set your paths and thresholds here — this is the only cell you should need to touch to run on a different video.

In [ ]:
VIDEO_PATH = "/content/sample_video.mp4"   # upload your test video here (Colab: Files sidebar -> upload)
LOG_PATH = "/content/plate_log.txt"
ANNOTATED_OUTPUT_PATH = "/content/annotated_output.mp4"  # optional visual sanity check

FRAME_SKIP = 5          # process every Nth frame (saves compute; tune based on your fps/vehicle speed)
PLATE_CONF_THRESH = 0.4 # minimum YOLO confidence to accept a plate detection
OCR_CONF_THRESH = 0.3   # minimum EasyOCR confidence to accept a text read
SAVE_ANNOTATED_VIDEO = True

## 4. Load models
- Plate detector: a pretrained YOLOv8 model fine-tuned specifically for license plate localization (generic COCO YOLO does NOT detect plates as a class, so a plate-specific model is required here).
- OCR: EasyOCR, English character set for now.

In [ ]:
plate_model = YOLO('keremberke/yolov8n-license-plate')
plate_model.overrides['conf'] = PLATE_CONF_THRESH
plate_model.overrides['iou'] = 0.45
plate_model.overrides['agnostic_nms'] = False
plate_model.overrides['max_det'] = 20

ocr_reader = easyocr.Reader(['en'], gpu=True)  # set gpu=False if no CUDA available

print("Models loaded.")

## 5. Helper functions
Small, testable pieces: cropping, cleaning OCR text, formatting timestamps, writing a log line.

In [ ]:
def crop_with_margin(frame, box, margin_ratio=0.05):
    """Crop a plate region from the frame with a small margin, clipped to frame bounds."""
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = box
    bw, bh = x2 - x1, y2 - y1
    mx, my = int(bw * margin_ratio), int(bh * margin_ratio)
    x1 = max(0, int(x1 - mx))
    y1 = max(0, int(y1 - my))
    x2 = min(w, int(x2 + mx))
    y2 = min(h, int(y2 + my))
    return frame[y1:y2, x1:x2]


def clean_plate_text(raw_text):
    """Basic cleanup: uppercase, strip whitespace, drop non-alphanumeric junk.
    This is a placeholder -- replace with a proper Nepali-plate format validator later."""
    text = raw_text.upper().strip()
    text = re.sub(r'[^A-Z0-9]', '', text)
    return text


def frame_index_to_video_timestamp(frame_idx, fps):
    """Convert a frame index to a HH:MM:SS.mmm position within the video."""
    seconds = frame_idx / fps if fps > 0 else 0
    td = timedelta(seconds=seconds)
    return str(td)


def log_detection(log_path, wall_clock_time, video_timestamp, frame_idx, plate_text, plate_conf, ocr_conf):
    """Append one structured line per detection to the log file."""
    line = (f"{wall_clock_time} | video_ts={video_timestamp} | frame={frame_idx} | "
            f"plate=\"{plate_text}\" | plate_conf={plate_conf:.2f} | ocr_conf={ocr_conf:.2f}\n")
    with open(log_path, 'a') as f:
        f.write(line)

## 6. Main pipeline
Reads the video frame-by-frame, samples every `FRAME_SKIP` frames, runs plate detection, crops, runs OCR, and logs each accepted read. Optionally writes an annotated video so you can visually sanity-check detections.

In [ ]:
def run_pipeline():
    # Fresh log file each run
    if os.path.exists(LOG_PATH):
        os.remove(LOG_PATH)

    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer = None
    if SAVE_ANNOTATED_VIDEO:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(ANNOTATED_OUTPUT_PATH, fourcc, fps / FRAME_SKIP, (width, height))

    frame_idx = 0
    detections_logged = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % FRAME_SKIP == 0:
            results = plate_model.predict(frame, verbose=False)
            boxes = results[0].boxes

            for box in boxes:
                plate_conf = float(box.conf[0])
                x1, y1, x2, y2 = box.xyxy[0].tolist()

                plate_crop = crop_with_margin(frame, (x1, y1, x2, y2))
                if plate_crop.size == 0:
                    continue

                ocr_results = ocr_reader.readtext(plate_crop)
                if not ocr_results:
                    continue

                # Take the highest-confidence text read in the crop
                best = max(ocr_results, key=lambda r: r[2])
                raw_text, ocr_conf = best[1], best[2]

                if ocr_conf < OCR_CONF_THRESH:
                    continue

                plate_text = clean_plate_text(raw_text)
                if not plate_text:
                    continue

                wall_clock_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                video_ts = frame_index_to_video_timestamp(frame_idx, fps)

                log_detection(LOG_PATH, wall_clock_time, video_ts, frame_idx,
                               plate_text, plate_conf, ocr_conf)
                detections_logged += 1

                if writer is not None:
                    cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
                    cv2.putText(frame, plate_text, (int(x1), int(y1) - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

            if writer is not None:
                writer.write(frame)

        frame_idx += 1

    cap.release()
    if writer is not None:
        writer.release()

    print(f"Done. Frames processed: {frame_idx}. Detections logged: {detections_logged}.")
    print(f"Log file: {LOG_PATH}")
    if SAVE_ANNOTATED_VIDEO:
        print(f"Annotated video: {ANNOTATED_OUTPUT_PATH}")


run_pipeline()

## 7. Inspect the log output

In [ ]:
with open(LOG_PATH, 'r') as f:
    print(f.read())

## Notes / next steps
- `keremberke/yolov8n-license-plate` is a generic plate detector — check its recall on Nepali plates specifically; it may miss embossed/older plates. This is exactly where you'll later fine-tune on your own dataset.
- EasyOCR here uses only the `en` character set — it won't handle Devanagari script at all. That's the next real piece of work.
- No deduplication yet — the same vehicle across consecutive sampled frames will log multiple times. Once this works end-to-end, add tracking (ByteTrack/DeepSORT) to log each vehicle once.
- `clean_plate_text` is a placeholder — replace with a real Nepali plate format validator once you know your target plate formats.
- Swapping OCR to TrOCR later: replace the `ocr_reader.readtext(...)` call with a TrOCR inference call inside the same loop; everything else (cropping, logging, thresholds) stays the same.